In [ ]:
"""GridSearchCv or RandommizedSearchCv dono ka main kaam model ke
best hyperparameters find karna hota hai

"""


import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

df = pd.read_csv("D:\Python Courses\ScikitLearn._Librarypy\Hyperparameter Tuning\heart.csv")

df.head()

x = df.iloc[:,0:-1]
y = df.iloc[:,-1]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

rf = RandomForestClassifier(max_samples=0.75,random_state=42) #max_sample ka mtlab k hr tree k pass kitny rows jaein gy
gb = GradientBoostingClassifier()
svc = SVC(probability=True, random_state=42)
lr = LogisticRegression()


rf.fit(x_train,y_train)
y_pred = rf.predict(x_test)
print(accuracy_score(y_test,y_pred))

from sklearn.model_selection import cross_val_score
np.mean(cross_val_score(RandomForestClassifier(max_samples=0.75,random_state=42),x,y,cv=10,scoring='accuracy'))

"""GridSearchCV"""
#Number of trees in random forest
n_estimators = [20,60,100,120]

#Number of features to consider at every split
max_features = [0.2,0.6,1.0]

#Maximum number of levels in tree
max_depth = [2,8,None]

#Number of samples
max_samples = [0.5,0.75,1.0]

param_grid = {
    'n_estimators' : n_estimators, #kitny decision trees rkhny hain
    'max_features' : max_features, #hr tree ko kitny columns dene hain
    'max_depth' : max_depth,
    'max_samples' : max_samples #hr tree ko rows ka kitny % sample data dena hai
}

from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(estimator=rf,
                       param_grid= param_grid,
                       cv = 5, #cv ka mtlab k isko 5 baar train kro
                       verbose= 2, #verbos sy output dikhy ga during the process
                       n_jobs=-1) # -1 use krny sy process thora fast ho jaye ga
# rf_grid.fit(x_train, y_train)
# print(rf_grid.best_params_)

# print(rf_grid.best_score_)

"""RandomSearchCV"""

#Number of trees in random forest
n_estimators = [20,60,100,120]

#Number of features to consider at every aplit
max_features = [0.2,0.6,1.0]

#Maximum number of levels in tree
max_depth = [2,8,None]

#Number of samples
max_samples = [0.5,0.75,1.0]

#Bootstap samples
bootstrap = [True,False] #bootstrap decide karta hai ke Random Forest ke trees ke liye training data ka random sample with replacement use karna hai ya nahi.

#Minimum number of samples required to split a node
min_samples_split = [2,5] #Agar kisi node mein 10 se kam samples hain, to usko further split nahi kiya jayega.

#Minimum number of samples required at each leaf node
min_samples_leaf = [1,2] #hr final leaf mein final kitny leafs hony chahiyye

param_grid = {
    'n_estimators' : n_estimators, #kitny decision trees rkhny hain
    'max_features' : max_features, #hr tree ko kitny columns dene hain
    'max_depth' : max_depth,
    'max_samples' : max_samples, #hr tree ko rows ka kitny % sample data dena hai
    'bootstrap' : bootstrap,
    'min_samples_split' : min_samples_split,
    'min_samples_leaf' : min_samples_leaf
}

from sklearn.model_selection import RandomizedSearchCV

rf_grid = RandomizedSearchCV(
                            estimator=rf,
                            param_distributions= param_grid,
                            cv=5,
                            verbose=2,
                            n_jobs=-1)
# rf_grid.fit(x_train,y_train)
# print(rf_grid.best_params_)
# print(rf_grid.best_score_)

from sklearn.ensemble import VotingClassifier
 
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('svc', svc),
        ('lr', lr)
    ],
    voting='soft'   # soft voting: har model ki probability ka average leta hai
                     # (hard voting ke liye 'hard' likhein - sirf majority class count hoti)
)
 
voting_clf.fit(x_train, y_train)
y_pred_voting = voting_clf.predict(x_test)
print(f"\nVoting Ensemble accuracy: {accuracy_score(y_test, y_pred_voting):.4f}")
